# Blindspots — proof of concept

A person says one thing. We show them the tetrad around it, with the corner they are
not looking at — **A+**, the constructive side of the opposing view — in front.

This notebook is the pre-MVP: no app, no server, no UI. It exists to answer two
questions with real output before anyone builds a backend:

1. Is the A+ the framework produces worth showing to a person?
2. Which of the two ways of producing it should the app run?
3. Can a person POLISH their way to the one that lands — react to a card, get a
   different or adjusted one, without it becoming a chat?

**Where it stands (2026-10-02, Sonnet 5, 40 utterances, the pipeline's own coherence
judge and auditors — `docs/dev-notes/antithesis-selection.md`, "the three doors" and
"The one-shot build"):** Path A draws a coherent first tetrad for 25/40 at ~7 s with a
genuine opposing position for 39/40. Path B — the Advisor's `anchor` — used to run a
staged pipeline (headline → antithesis ladder → aspects for a pair never seen whole) that
drew 14/40 coherent at ~55 s with mostly strawman antitheses; since `fac4909` it reasons
the tetrad in ONE call, the same call Path A makes, and then classifies, scores, grounds,
validates and persists it: 21/40 and 21/40 coherent on two generations, 36–38/40 genuine
antitheses, ~43 s. The two paths now draw the same tetrad; Path B adds the graph.
So the pre-MVP runs **Path A** for the screen, and Path B is what a "your recurring
blindspots" screen would run, because it is the one with scores and memory.

**Path A** asks the tool-less `Consultant` for the view — one structured turn, no
database. **Path B** keeps the utterance as an `Input` and runs the one-shot build
(`SketchTetrad`: one reasoning call finds the thesis IN the utterance and writes the
tetrad; `IntroducePolarity` and `ExpandPolarity` score, dedup, ground, validate and
commit it).


In [ ]:
from dialectical_framework.dialectical_reasoning import DialecticalReasoning
from dialectical_framework.settings import Settings

# DI setup, once. Reads .env (host obligation 1 — see examples/advisor_chat.py).
container = DialecticalReasoning.setup(Settings.from_env())
print("model:", container.settings().ai_model)

In [ ]:
UTTERANCE = "I should quit my job and start my own company."


## The renderer

Both paths return `graph/views.py::PerspectiveView`. A consumer app never shows
framework terminology, so the view goes through `without_terminology()` — the same
projection the Advisor applies under `hides_terminology` — and the card names the
corners in plain words. `intent` is the reading (the two axes the tetrad was read
along); `complete` says whether all six corners exist.


In [ ]:
from IPython.display import Markdown, display

from dialectical_framework.graph.views import PerspectiveView


def render(view: PerspectiveView, title: str = "") -> None:
    """One card per perspective: A+ first, the two traps after, the poles as context."""
    v = view.without_terminology()
    text = lambda pole: pole.text if pole else "_(not drawn)_"
    lines = [f"### {title}" if title else ""]
    lines += [
        f"**What you might be missing:** {text(v.a_plus)}",
        "",
        f"- Your position: {text(v.t)}",
        f"- What it stands against: {text(v.a)}",
        f"- Your position, developed well: {text(v.t_plus)}",
        "",
        f"- Trap on your side: {text(v.t_minus)}",
        f"- Trap on the other side: {text(v.a_minus)}",
    ]
    if v.intent:
        lines += ["", f"_{v.intent}_"]
    if not v.complete:
        lines += ["", "_(unfinished tetrad — some corners were not drawn)_"]
    display(Markdown("\n".join(lines)))

## Path A — the Consultant's view turn (no graph) — the pre-MVP path

The utterance is the only message in the conversation; `exploration_view(focus=…)` is
one structured, thinking turn that builds the tetrad and returns it. Nothing is
checked and nothing is kept. Typically 5–25 s.


In [ ]:
from dialectical_framework.agents.apps import COUNSELOR_PERSONA
from dialectical_framework.agents.consultant.consultant import Consultant

FOCUS = (
    "the perspective for what I just said: my position, what it stands against, "
    "and above all the constructive side of that opposing view that I am not seeing"
)


async def blindspot_via_consultant(text: str) -> PerspectiveView | None:
    head = Consultant(app_preamble=COUNSELOR_PERSONA, messages=[{"role": "user", "content": text}])
    view = await head.exploration_view(focus=FOCUS)
    return view.perspectives[0] if view.perspectives else None


a = await blindspot_via_consultant(UTTERANCE)
render(a, "Path A — Consultant") if a else print("nothing drawn")

## Polishing — Path A

The promise of the app is not "a tetrad" but the one that lands: *"that's a good one,
I hadn't seen it."* So the first tetrad is a draft, and the person's reaction steers the
next. This is not a chat: the person never talks to a counsellor, they react to a card —
"too obvious", "it's not about money, it's about his dignity", "show me two more".

On Path A the Consultant already holds the conversation: `exploration_view()` records
what it drew, in words, so the next call with the reaction as `focus` draws AGAINST what
was shown. Nothing is kept beyond `head.messages`.


In [ ]:
class PolishA:
    """One Consultant, many cards. Each reaction becomes the next view's focus."""

    def __init__(self, text: str) -> None:
        self.head = Consultant(app_preamble=COUNSELOR_PERSONA, messages=[{"role": "user", "content": text}])
        self.cards: list[PerspectiveView] = []

    async def first(self) -> list[PerspectiveView]:
        return await self._draw(FOCUS)

    async def react(self, reaction: str, how_many: int = 1) -> list[PerspectiveView]:
        """`reaction` in the person's words; `how_many` tetrads to draw this time (1–4)."""
        count = {1: "one tension", 2: "two different tensions", 3: "three different tensions"}.get(how_many, "up to four different tensions")
        focus = (
            f"The person reacted to what was shown: \"{reaction}\". Draw {count} they have NOT "
            "seen yet, in the light of that reaction — a different tension where they ask for one, "
            "the same tension re-read where they ask to adjust it — and above all the constructive "
            "side of the opposing view they are not seeing."
        )
        return await self._draw(focus)

    async def _draw(self, focus: str) -> list[PerspectiveView]:
        view = await self.head.exploration_view(focus=focus)
        self.cards.extend(view.perspectives)
        for i, pv in enumerate(view.perspectives, 1):
            render(pv, f"Path A — card {len(self.cards) - len(view.perspectives) + i}")
        if not view.perspectives:
            print("nothing drawn")
        return list(view.perspectives)


polish_a = PolishA(UTTERANCE)
await polish_a.first()


In [ ]:
# The person is not convinced. React, and get a different one — or several.
await polish_a.react("Too obvious — I know the security argument. What am I REALLY not seeing?", how_many=2)


## Path B — the one-shot build into the graph — scores and memory; not the pre-MVP path

What the Advisor's `anchor` runs since `fac4909`, called headlessly inside a Case. The
utterance is NOT the thesis: it is the person's material, kept verbatim as an `Input`
(`AddInput`), and the build's one reasoning call finds the position in it — the thesis
is distilled, not pasted. Then the graph's checks run on what came out: both poles are
classified against the systemic taxonomy, the antithesis scored (HS, Mode, Arousal), the
four aspects scored (complementarity), the tetrad grounded in the utterance's particulars,
validated (coherence + the empirical inequalities), and committed — so a second
utterance from the same person lands next to the first, traces to its own Input, and
`exploration_view()` becomes their map of recurring blindspots.

The host owns the Case (the framework never creates one) and every call runs inside
`scope(sid)`. ~43 s today (~7 s is the reasoning call, the rest the checks); the
`sid:progress` channel exists to show the person what is happening meanwhile.


In [ ]:
from dialectical_framework.agents.analyst.skills.sketch_tetrad import SketchTetrad
from dialectical_framework.concerns.add_input import capture_input
from dialectical_framework.graph.nodes.case import Case
from dialectical_framework.graph.scope_context import scope
from dialectical_framework.graph.views import exploration_view, perspective_view

# One Case per person. Kept across calls: that is where the memory lives.
case = Case()
case.commit()
print("sid:", case.sid)


async def blindspot_via_pipeline(text: str) -> PerspectiveView | None:
    with scope(case.sid):
        # Whatever the person typed or pasted is MATERIAL, not the thesis: one
        # Input (added and digested), which the build reads and distils the
        # position from. The tetrad traces to it.
        source = await capture_input(text)
        skill = SketchTetrad(input_hashes=[source.hash])
        perspectives = await skill.resolve()
        if not perspectives:
            print("nothing built:", skill.report.summary)
            return None
        return perspective_view(perspectives[0])


b = await blindspot_via_pipeline(UTTERANCE)
render(b, "Path B — one-shot build") if b else None


## Polishing — Path B

Same loop, with memory. Every card is a Perspective in the Case; a card the person
rejects is **discarded with their reason** (the graph's soft-mark — nothing is deleted,
and the Advisor would deprioritise it, not show it again as new), and the next build gets
that reason as `context`, the particulars slot every prompt composes ahead of the
material. The map at the end shows the whole polishing journey, discards included —
which is what a "your blindspots" screen is made of.


In [ ]:
class PolishB:
    """One Case, one Input, many builds. Rejections are kept, as reasons on the graph."""

    def __init__(self, text: str) -> None:
        self.text = text
        self.source = None
        self.cards: list = []  # (Perspective, PerspectiveView)
        self.reasons: list[str] = []

    async def first(self):
        with scope(case.sid):
            self.source = await capture_input(self.text)
        return await self._build(context="")

    async def react(self, reaction: str, reject: bool = True):
        """`reject=True` discards the last card with the reaction as its reason;
        `reject=False` keeps it and asks for another beside it."""
        with scope(case.sid):
            if reject and self.cards:
                last_pp, last_view = self.cards[-1]
                last_pp.discarded = reaction          # the person's words, kept on the node
                last_pp.save()
        self.reasons.append(reaction)
        shown = "; ".join(f"«{pv.t.text} vs {pv.a.text}»" for _pp, pv in self.cards if pv.t and pv.a)
        context = (
            f"Already shown to the person: {shown}. Their reaction: \"{reaction}\". "
            "Build a DIFFERENT tension where they ask for one, or the same one re-read where they "
            "ask to adjust it — never a restatement of what was shown."
        )
        return await self._build(context=context)

    async def _build(self, context: str):
        with scope(case.sid):
            skill = SketchTetrad(input_hashes=[self.source.hash], context=context)
            pps = await skill.resolve()
            if not pps:
                print("nothing built:", skill.report.summary)
                return None
            view = perspective_view(pps[0])
        self.cards.append((pps[0], view))
        render(view, f"Path B — card {len(self.cards)}")
        return view


polish_b = PolishB(UTTERANCE)
await polish_b.first()


In [ ]:
await polish_b.react("Too obvious — I know the security argument. What am I REALLY not seeing?")


### What Path B knows that Path A cannot say

The same view, before `without_terminology()`: the antithesis's HS (how well the
opposition sits against the taxonomy apex), the complementarity of each corner, the
tetrad's SP, and the validation verdict. None of this exists on Path A — its numbers
are `None` because nothing scored them, not because they are zero.


In [ ]:
if b:
    print("HS(A)        :", b.a.hs)
    print("Ks per corner:", {n: round(p.ks, 2) for n, p in b.poles.items() if p.ks is not None})
    print("metrics      :", b.metrics)
    print("validation   :", b.validation)

### The map after polishing (Path B only)

Two records of the journey. The **view** (`exploration_view()`) shows the standing
cards only — a rejected framing is set aside on the graph, with the person's reason on
the node, and the active view does not show it again as if it were new (the same rule
the Advisor's context follows). The **journey** — every card and every reaction — is
the host's own record, `polish_b.cards` and `polish_b.reasons`, which a "your
blindspots" screen would keep alongside the map.


In [ ]:
with scope(case.sid):
    the_map = exploration_view()
print(f"standing: {len(the_map.perspectives)} card(s); set aside: {sum(1 for pp, _ in polish_b.cards if pp.discarded)}")
for i, p in enumerate(the_map.perspectives, 1):
    render(p, f"standing #{i}")
for (pp, pv), reason in zip(polish_b.cards, polish_b.reasons):
    if pp.discarded:
        print(f"set aside — «{pv.t.text} vs {pv.a.text}» — because: {reason}")


## Where this goes

- **Re-measure before trusting it.** `tests/e2e/probe_view_turn_ask.py` draws the 40
  utterances the way Path A does and audits them; `tests/e2e/probe_tetrad_quality.py`
  (`TETRAD_PROBE_MODE=oneshot`) runs Path B's build on the same 40. Every item is kept
  as JSON; the figures above are two generations, not a result.
- **Backend**: a small HTTP service hosting the chosen path — `POST /blindspot {text}` →
  `PerspectiveView.to_dict()`. Host obligations are the documented ones (Case per
  person, `scope(sid)` per request, `principal="human"`, drain at shutdown).
- **Not a tension**: a bare question may draw nothing. The app needs a graceful
  "nothing to draw yet" state; Path A returns an empty view, Path B reports no anchor.
- **Unmeasured**: A+ quality on either path against real users' phrasing. The probe
  is the first sample, not a result.
